# É uma técnica que envia determinado assunto para um determinado especialista

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain.chains.llm import LLMChain

from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain.chains.router.multi_prompt_prompt import MULTI_PROMPT_ROUTER_TEMPLATE

# Carregar variáveis de ambiente
load_dotenv()
chat = ChatOpenAI(api_key = os.getenv("OPENAI_API_KEY"),
                  model="gpt-5-mini", temperature=0.7)

### Prompts

In [4]:
social_media_template = ChatPromptTemplate.from_template("""
Você é um especialista em marketing de mídias sociais.
Sua tarefa é criar títulos e slogans curtos e envolventes para posts.
O título deve ser otimizado para plataformas como Instagram e LinkedIn.

Aqui está a solicitação: {input}""")

ecommerce_template = ChatPromptTemplate.from_template("""
Você é um redator de e-commerce.
Sua especialidade é escrever descrições de produtos que destacam seus benefícios,
características e que são persuasivas para o cliente.

Aqui está a solicitação: {input}""")

video_template = ChatPromptTemplate.from_template("""
Você é um criador de conteúdo para vídeos curtos (Reels, TikTok).
Sua missão é criar roteiros concisos, com ganchos fortes,
para capturar a atenção do público nos primeiros segundos.

Aqui está a solicitação: {input}""")


### Agrupando Informações ao Roteador

In [5]:
prompt_infos = [
    {"name": "SocialMediaExpert",
     "description": "Ideal para criar slogans e títulos para redes sociais.",
     "prompt_template": social_media_template
    },
    {"name": "EcommerceExpert",
     "description": "Ideal para escrever descrições de produtos para lojas online.",
     "prompt_template": ecommerce_template
    },
    {"name": "VideoExpert",
     "description": "Ideal para gerar roteiros para vídeos curtos e virais.",
     "prompt_template": video_template
    },
]

### Criando as Chains

In [6]:
chains_destino = {}

for info in prompt_infos:
    chain = LLMChain(llm=chat, prompt=info["prompt_template"], verbose=True)
    chains_destino[info["name"]] = chain
    
print("Cadeias de destino criadas:", list(chains_destino.keys()))

Cadeias de destino criadas: ['SocialMediaExpert', 'EcommerceExpert', 'VideoExpert']


C:\Users\rodri\AppData\Local\Temp\ipykernel_8676\1588908755.py:4: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=chat, prompt=info["prompt_template"], verbose=True)


In [7]:
# Preparar a lista de destinos para o roteador
destinos = [f'{p["name"]}: {p["description"]}' for p in prompt_infos]
destinos_str = '\n'.join(destinos)
print("\n--- Destinos para o Roteador ---\n", destinos_str)


--- Destinos para o Roteador ---
 SocialMediaExpert: Ideal para criar slogans e títulos para redes sociais.
EcommerceExpert: Ideal para escrever descrições de produtos para lojas online.
VideoExpert: Ideal para gerar roteiros para vídeos curtos e virais.


### Configurar o Roteador

In [8]:
# Criar o template do roteador
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinos_str
)

router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser()
)

# Criar a cadeia de roteamento
router_chain = LLMRouterChain.from_llm(chat, router_prompt, verbose=True)


### Juntando em MultiPromptChain

In [9]:
default_prompt = ChatPromptTemplate.from_template("Responda de forma genérica a esta solicitação: {input}")
default_chain = LLMChain(llm=chat, prompt=default_prompt, verbose=True)

# Unir tudo na MultiPromptChain
chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chains=chains_destino,
    default_chain=default_chain,
    verbose=True
)


C:\Users\rodri\AppData\Local\Temp\ipykernel_8676\2054119591.py:5: LangChainDeprecationWarning: Please see migration guide here for recommended implementation: https://python.langchain.com/docs/versions/migrating_chains/multi_prompt_chain/
  chain = MultiPromptChain(


In [10]:
print("\n--- Teste 1: Solicitação para Redes Sociais ---")
chain.invoke({"input": "Crie um post sobre o lançamento de um novo smartphone"})


--- Teste 1: Solicitação para Redes Sociais ---


> Entering new MultiPromptChain chain...


> Entering new LLMRouterChain chain...

> Finished chain.
SocialMediaExpert: {'input': 'Crie um post para redes sociais sobre o lançamento de um novo smartphone. Inclua: 1) um título chamativo; 2) uma legenda curta (2-3 frases) com tom entusiasmado; 3) uma chamada para ação para pré-venda; 4) 3 sugestões de hashtags; público-alvo: jovens adultos (18-35); plataforma: Instagram.'}

> Entering new LLMChain chain...
Prompt after formatting:
Human: 
Você é um especialista em marketing de mídias sociais.
Sua tarefa é criar títulos e slogans curtos e envolventes para posts.
O título deve ser otimizado para plataformas como Instagram e LinkedIn.

Aqui está a solicitação: Crie um post para redes sociais sobre o lançamento de um novo smartphone. Inclua: 1) um título chamativo; 2) uma legenda curta (2-3 frases) com tom entusiasmado; 3) uma chamada para ação para pré-venda; 4) 3 sugestões de hashtags; púb

{'input': 'Crie um post para redes sociais sobre o lançamento de um novo smartphone. Inclua: 1) um título chamativo; 2) uma legenda curta (2-3 frases) com tom entusiasmado; 3) uma chamada para ação para pré-venda; 4) 3 sugestões de hashtags; público-alvo: jovens adultos (18-35); plataforma: Instagram.',
 'text': 'Título:\nO futuro cabe na sua mão — descubra o novo smartphone 🚀\n\nLegenda (2–3 frases, entusiasmada):\nPronto para elevar seu feed e suas jogatinas? Com câmera profissional, bateria para o dia todo e performance ultra‑rápida, ele acompanha cada momento do seu dia. Design leve e recursos pensados pra quem cria, conecta e vive intensamente!\n\nChamada para ação (pré‑venda):\nPré‑venda aberta — garanta o seu com vantagens exclusivas! Reserve já pelo link na bio. ⏳🔥\n\n3 sugestões de hashtags:\n#PréVenda #SmartphoneNovo #VidaConectada'}

In [11]:
print("\n--- Teste 2: Solicitação para E-commerce ---")
chain.invoke({"input": "Escreva uma descrição para um notebook gamer de alta performance."})


--- Teste 2: Solicitação para E-commerce ---


> Entering new MultiPromptChain chain...


> Entering new LLMRouterChain chain...

> Finished chain.
EcommerceExpert: {'input': 'Escreva uma descrição persuasiva e otimizada para loja online (150–200 palavras) para um notebook gamer de alta performance. Destaque desempenho (CPU e GPU de última geração), memória e armazenamento rápidos (mín. 16GB RAM e SSD), tela com alta taxa de atualização, sistema de resfriamento eficiente, design robusto com iluminação RGB, boas opções de conectividade e autonomia de bateria. Foque nos benefícios para jogadores (frames estáveis, multitarefa, portabilidade), use português (tom entusiasmado e profissional) e termine com uma chamada para ação (ex.: "Compre agora e leve seu jogo ao próximo nível").'}

> Entering new LLMChain chain...
Prompt after formatting:
Human: 
Você é um redator de e-commerce.
Sua especialidade é escrever descrições de produtos que destacam seus benefícios,
características e que são p

{'input': 'Escreva uma descrição persuasiva e otimizada para loja online (150–200 palavras) para um notebook gamer de alta performance. Destaque desempenho (CPU e GPU de última geração), memória e armazenamento rápidos (mín. 16GB RAM e SSD), tela com alta taxa de atualização, sistema de resfriamento eficiente, design robusto com iluminação RGB, boas opções de conectividade e autonomia de bateria. Foque nos benefícios para jogadores (frames estáveis, multitarefa, portabilidade), use português (tom entusiasmado e profissional) e termine com uma chamada para ação (ex.: "Compre agora e leve seu jogo ao próximo nível").',
 'text': 'Apresentamos o notebook gamer de alta performance projetado para jogadores que não aceitam limites. Equipado com CPU e GPU de última geração, ele entrega potência suficiente para rodar jogos AAA com frames estáveis e gráficos impressionantes. Com no mínimo 16 GB de RAM (expansível) e SSD NVMe ultrarrápido, o carregamento de jogos, streaming e multitarefa acontece

In [12]:
print("\n--- Teste 3: Solicitação para Vídeos ---")
chain.invoke({"input": "Elabore um roteiro para um vídeo de 15 segundos sobre um produto de limpeza."})


--- Teste 3: Solicitação para Vídeos ---


> Entering new MultiPromptChain chain...


> Entering new LLMRouterChain chain...

> Finished chain.
VideoExpert: {'input': 'Elabore um roteiro em português para um vídeo vertical de 15 segundos sobre um produto de limpeza (por exemplo, removedor multiuso). Público-alvo: donas de casa e jovens adultos. Tom: enérgico, confiante e convidativo. Estruture o roteiro em cenas com duração em segundos, falas/sobreposição de texto, descrições visuais e sugestões de som/efeitos, e inclua uma chamada para ação clara (ex.: "Compre agora" ou "Saiba mais").'}

> Entering new LLMChain chain...
Prompt after formatting:
Human: 
Você é um criador de conteúdo para vídeos curtos (Reels, TikTok).
Sua missão é criar roteiros concisos, com ganchos fortes,
para capturar a atenção do público nos primeiros segundos.

Aqui está a solicitação: Elabore um roteiro em português para um vídeo vertical de 15 segundos sobre um produto de limpeza (por exemplo, removedor multiu

{'input': 'Elabore um roteiro em português para um vídeo vertical de 15 segundos sobre um produto de limpeza (por exemplo, removedor multiuso). Público-alvo: donas de casa e jovens adultos. Tom: enérgico, confiante e convidativo. Estruture o roteiro em cenas com duração em segundos, falas/sobreposição de texto, descrições visuais e sugestões de som/efeitos, e inclua uma chamada para ação clara (ex.: "Compre agora" ou "Saiba mais").',
 'text': 'Roteiro — Vídeo vertical (15s) — Removedor Multiuso\nPúblico: donas de casa e jovens adultos\nTom: enérgico, confiante, convidativo\n\nCena 1 (0:00–0:02) — Gancho imediato\n- Falas / Sobreposição de texto: “Cansada da sujeira impossível?”\n- Visual: Close-up em uma mancha difícil (fogão engordurado ou mancha na bancada). Pessoa suspira/franze a testa.\n- Som / Efeitos: Som de record scratch rápido + batida de música enérgica entrando.\n- Observação: rosto expressivo para conectar com donas de casa e jovens.\n\nCena 2 (0:02–0:06) — Apresenta o pro

In [13]:
print("\n--- Teste 4: Solicitação Genérica ---")
chain.invoke({"input": "Qual é a capital da França?"})


--- Teste 4: Solicitação Genérica ---


> Entering new MultiPromptChain chain...


> Entering new LLMRouterChain chain...

> Finished chain.
None: {'input': 'Qual é a capital da França?'}

> Entering new LLMChain chain...
Prompt after formatting:
Human: Responda de forma genérica a esta solicitação: Qual é a capital da França?

> Finished chain.

> Finished chain.


{'input': 'Qual é a capital da França?',
 'text': 'Resposta genérica: "A capital do país em questão é [nome da cidade]."\n\nExemplo aplicado: "A capital da França é Paris."'}